# CWR encounter data extraction reconnaissance

**Target:** <https://www.whaleresearch.com/encounters-map-2026>  
**Purpose:** identify the least brittle source of Center for Whale Research (CWR) encounter records and build a small proof of concept. This is not a production ingestion pipeline.

## Guardrails

- Treat CWR as the publisher and preserve its page URL and Atlist record IDs as provenance.
- Do not infer absent values. In particular, an Atlist marker coordinate is not automatically an encounter start or end coordinate.
- Keep network captures bounded; do not save credentials, cookies, access tokens, or large static/binary responses.
- Re-check CWR's terms and obtain any needed permission before turning this reconnaissance into a recurring ingestion process or redistributing narrative/photo content.

## Observed on 2026-08-18

The CWR page is a Wix page whose encounter map is an iframe served by Atlist. The iframe URL is `https://my.atlist.com/map/4fdb0211-7443-48fb-9334-2c401932b4ba?share=true`. The iframe requests public JSON from `api.atlist.com/v1/map/{map_id}/...`, including `markers`, `categories`, `fields`, `images`, `circles`, `polygons`, `geoPolygons`, and `lines`. The direct `markers` response is the leading 2026 extraction candidate.


In [ ]:
from __future__ import annotations

import asyncio
import html
import json
import re
from datetime import datetime
from html.parser import HTMLParser
from pathlib import Path
from typing import Any
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import pandas as pd
import requests
from IPython.display import display


In [ ]:
TARGET_PAGE = "https://www.whaleresearch.com/encounters-map-2026"
ATLIST_MAP_ID = "4fdb0211-7443-48fb-9334-2c401932b4ba"
ATLIST_IFRAME_URL = f"https://my.atlist.com/map/{ATLIST_MAP_ID}?share=true"
ATLIST_API_ROOT = f"https://api.atlist.com/v1/map/{ATLIST_MAP_ID}"
ATLIST_ENDPOINTS = {
    name: f"{ATLIST_API_ROOT}/{name}"
    for name in ("fields", "markers", "categories", "images", "circles", "polygons", "geoPolygons", "lines")
}

import os
NOTEBOOK_DIR = Path(os.environ["MARINE_MAMMALS_CWR_OUTPUT_ROOT"]).expanduser().resolve()

EXPLORATION_DIR = NOTEBOOK_DIR / "exploration"
SAMPLE_DIR = EXPLORATION_DIR / "samples"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"

REQUEST_TIMEOUT_SECONDS = 30
POC_LIMIT = 10  # keep the proof of concept intentionally small (5-20)
RUN_BROWSER_CAPTURE = False  # set True after installing Playwright/Chromium
WRITE_SANITIZED_SAMPLE = False

session = requests.Session()
session.headers.update({"User-Agent": "OrcaCast-CWR-reconnaissance/0.1 (research; bounded requests)"})

print({"target_page": TARGET_PAGE, "atlist_map_id": ATLIST_MAP_ID, "notebook_dir": str(NOTEBOOK_DIR.resolve())})


## 1. Inspect CWR page HTML

This inventory checks raw HTML for iframes, scripts, links, embedded JSON, and high-signal terms. It does not assume that the raw Wix page contains the encounter records.


In [ ]:
class PageInventoryParser(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.iframes: list[dict[str, str | None]] = []
        self.scripts: list[dict[str, str | None]] = []
        self.links: list[dict[str, str | None]] = []
        self._current_script: dict[str, Any] | None = None

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        values = dict(attrs)
        if tag == "iframe":
            self.iframes.append({key: values.get(key) for key in ("src", "title", "name", "id")})
        elif tag == "script":
            item: dict[str, Any] = {key: values.get(key) for key in ("src", "type", "id")}
            item["inline_text"] = []
            self.scripts.append(item)
            self._current_script = item
        elif tag == "a" and values.get("href"):
            self.links.append({"href": values.get("href")})

    def handle_data(self, data: str) -> None:
        if self._current_script is not None and not self._current_script.get("src"):
            self._current_script["inline_text"].append(data)

    def handle_endtag(self, tag: str) -> None:
        if tag == "script":
            self._current_script = None


def inspect_page_html(url: str) -> tuple[str, dict[str, Any]]:
    response = session.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    parser = PageInventoryParser()
    parser.feed(response.text)

    for script in parser.scripts:
        script["inline_text"] = "".join(script["inline_text"])[:2_000]

    keyword_pattern = re.compile(r"https?://[^\s\"'<>]+(?:api|json|geojson|map|dataset|collection|wix|query|encounter)[^\s\"'<>]*", re.I)
    inventory = {
        "status": response.status_code,
        "content_type": response.headers.get("content-type"),
        "html_bytes": len(response.content),
        "iframes": parser.iframes,
        "script_srcs": [item["src"] for item in parser.scripts if item.get("src")],
        "json_script_ids": [item.get("id") for item in parser.scripts if "json" in (item.get("type") or "").lower()],
        "keyword_urls": sorted({url[:500] for url in keyword_pattern.findall(response.text)})[:30],
    }
    return response.text, inventory


try:
    page_html, page_inventory = inspect_page_html(TARGET_PAGE)
    display(page_inventory)
except requests.RequestException as exc:
    page_html, page_inventory = "", {"error": repr(exc)}
    print("CWR page inspection unavailable:", exc)


## 2. Capture browser network traffic

The helper below is the rerunnable equivalent of a temporary `discover_cwr_sources.py` script. It records bounded metadata and small text/JSON previews for documents, XHR, and fetch responses. It excludes large/binary bodies and redacts sensitive-looking query parameters.

Run it first against `TARGET_PAGE`, then directly against `ATLIST_IFRAME_URL` to reduce Wix noise. If Jupyter cannot launch Chromium in a sandboxed environment, run the notebook with the repository's normal notebook execution permissions.


In [ ]:
SENSITIVE_QUERY_KEYS = {"access_token", "api_key", "apikey", "auth", "authorization", "credential", "key", "password", "session", "signature", "token"}
INTERESTING_URL_RE = re.compile(r"api|json|geojson|map|dataset|collection|query|marker|encounter|atlist|wix|graphql", re.I)
TEXT_CONTENT_TYPES = ("application/json", "text/json", "application/geo+json", "text/plain", "text/html", "application/graphql-response+json")
MAX_CAPTURE_BODY_BYTES = 1_000_000
MAX_CAPTURE_PREVIEW_CHARS = 20_000


def sanitize_url(url: str) -> str:
    parts = urlsplit(url)
    query = []
    for key, value in parse_qsl(parts.query, keep_blank_values=True):
        query.append((key, "<redacted>" if key.lower() in SENSITIVE_QUERY_KEYS else value))
    return urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(query), parts.fragment))


def sanitize_payload(payload: str | None) -> str | None:
    if payload is None:
        return None
    value = payload[:4_000]
    return re.sub(r'(?i)(token|authorization|password|signature|credential)([=\"\s:]+)[^&\"\s,}]+', r'\1\2<redacted>', value)


async def capture_browser_traffic(start_url: str, wait_seconds: int = 7) -> dict[str, Any]:
    try:
        from playwright.async_api import async_playwright
    except ImportError as exc:
        raise RuntimeError("Install Playwright in the notebook environment before enabling RUN_BROWSER_CAPTURE.") from exc

    observations: list[dict[str, Any]] = []
    pending: set[asyncio.Task[Any]] = set()

    async def inspect_response(response: Any) -> None:
        request = response.request
        content_type = (await response.header_value("content-type") or "").lower()
        interesting = (
            request.resource_type in {"document", "xhr", "fetch"}
            or bool(INTERESTING_URL_RE.search(response.url))
            or any(value in content_type for value in TEXT_CONTENT_TYPES)
        )
        if not interesting:
            return

        item: dict[str, Any] = {
            "url": sanitize_url(response.url),
            "method": request.method,
            "status": response.status,
            "resource_type": request.resource_type,
            "content_type": content_type,
            "request_payload": sanitize_payload(request.post_data),
            "body_preview": None,
            "body_bytes": None,
            "capture_error": None,
        }
        if any(value in content_type for value in TEXT_CONTENT_TYPES):
            try:
                body = await response.body()
                item["body_bytes"] = len(body)
                if len(body) <= MAX_CAPTURE_BODY_BYTES:
                    item["body_preview"] = body.decode("utf-8", errors="replace")[:MAX_CAPTURE_PREVIEW_CHARS]
            except Exception as exc:  # response may be gone by the time an event task reads it
                item["capture_error"] = repr(exc)
        observations.append(item)

    def schedule_response(response: Any) -> None:
        task = asyncio.create_task(inspect_response(response))
        pending.add(task)
        task.add_done_callback(pending.discard)

    async with async_playwright() as playwright:
        try:
            browser = await playwright.chromium.launch(headless=True, channel="chrome")
        except Exception:
            browser = await playwright.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()
        page.on("response", schedule_response)
        await page.goto(start_url, wait_until="domcontentloaded", timeout=60_000)
        await page.wait_for_timeout(wait_seconds * 1_000)
        frame_urls = [frame.url for frame in page.frames]
        if pending:
            await asyncio.gather(*list(pending), return_exceptions=True)
        await browser.close()

    observations.sort(key=lambda item: (item["url"], item["method"]))
    return {"start_url": start_url, "frames": frame_urls, "responses": observations}


In [ ]:
browser_capture: dict[str, Any] | None = None
if RUN_BROWSER_CAPTURE:
    browser_capture = await capture_browser_traffic(TARGET_PAGE)
    atlist_capture = await capture_browser_traffic(ATLIST_IFRAME_URL)
    browser_capture["atlist_direct"] = atlist_capture
    print("CWR frames:", browser_capture["frames"])
    display(pd.DataFrame(browser_capture["responses"]).drop(columns=["body_preview"], errors="ignore"))
    display(pd.DataFrame(atlist_capture["responses"]).drop(columns=["body_preview"], errors="ignore"))
else:
    print("Browser capture is defined but not run. Set RUN_BROWSER_CAPTURE = True to refresh it.")


## 3. Reproduce the Atlist source without a browser

These requests intentionally use a fresh session with no authentication or cookies. The response should be one complete JSON envelope; no pagination parameters were observed in the browser request. Do not freeze a record count because the 2026 map is still being updated.


In [ ]:
def public_json_get(url: str) -> tuple[dict[str, Any] | list[Any], dict[str, Any]]:
    clean_session = requests.Session()
    clean_session.headers.update({"User-Agent": session.headers["User-Agent"]})
    response = clean_session.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    payload = response.json()
    evidence = {
        "url": response.url,
        "status": response.status_code,
        "content_type": response.headers.get("content-type"),
        "content_bytes": len(response.content),
        "access_control_allow_origin": response.headers.get("access-control-allow-origin"),
        "authorization_sent": "Authorization" in response.request.headers,
        "cookie_sent": "Cookie" in response.request.headers,
        "request_body_sent": response.request.body is not None,
    }
    return payload, evidence


map_metadata, fields_evidence = public_json_get(ATLIST_ENDPOINTS["fields"])
markers_envelope, markers_evidence = public_json_get(ATLIST_ENDPOINTS["markers"])
categories_envelope, categories_evidence = public_json_get(ATLIST_ENDPOINTS["categories"])

markers = markers_envelope.get("markers", []) if isinstance(markers_envelope, dict) else []
categories = categories_envelope.get("items", []) if isinstance(categories_envelope, dict) else []

display(pd.DataFrame([fields_evidence, markers_evidence, categories_evidence]))
print({
    "map_name": map_metadata.get("name") if isinstance(map_metadata, dict) else None,
    "map_updated_at": map_metadata.get("updatedAt") if isinstance(map_metadata, dict) else None,
    "marker_count": len(markers),
    "category_count": len(categories),
    "marker_keys": sorted(markers[0]) if markers else [],
})


### Direct-source interpretation

- `/markers` supplies Atlist UUID, marker title, one signed decimal latitude/longitude pair, HTML notes, tags, category linkage, timestamps, and image/order metadata.
- The HTML notes contain labeled values such as encounter summary, observation begin/end, pods, IDs encountered, and location description. Field presence varies by encounter.
- The single marker coordinate has no explicit start/end role. Preserve it as `map_lat`/`map_lon` with `coordinate_role = map_marker_unspecified`; leave all start/end coordinate fields null unless a separate authoritative source defines them.
- Atlist is the public presentation datastore for the 2026 map, while CWR remains the publisher/source. Preserve both URLs and both source identities.


## 4. Parse labeled note fields and coordinates

The parser retains the original HTML in memory, extracts only explicitly labeled fields, and keeps source text for times because no time zone is stated. It does not treat a positive longitude without an `E`/`W` hemisphere as unambiguous.


In [ ]:
NOTE_LABELS = ("EncSummary", "ObservBegin", "ObservEnd", "Vessel", "Staff", "Other Observers", "Pods", "IDsEncountered", "LocationDescr")
NOTE_LABEL_PATTERN = re.compile(rf"(?im)^({'|'.join(re.escape(label) for label in NOTE_LABELS)})\s*:\s*")
ENCOUNTER_NAME_PATTERN = re.compile(r"^Encounter\s+#(?P<number>[^-]+?)\s+-\s+(?P<date>.+?)\s*$", re.I)
ECOTYPE_TAGS = {"Southern Resident Killer Whales", "Northern Resident Killer Whales", "Bigg's Killer Whales"}


def html_to_lines(value: str | None) -> str:
    if not value:
        return ""
    text = re.sub(r"(?i)<br\s*/?>", "\n", value)
    text = re.sub(r"(?i)</p\s*>", "\n", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = html.unescape(text).replace("\xa0", " ")
    return "\n".join(line.strip() for line in text.splitlines() if line.strip())


def parse_labeled_notes(notes_html: str | None) -> dict[str, str]:
    text = html_to_lines(notes_html)
    matches = list(NOTE_LABEL_PATTERN.finditer(text))
    parsed: dict[str, str] = {}
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        parsed[match.group(1)] = text[match.end():end].strip()
    return parsed


def split_values(value: str | None) -> list[str]:
    if not value:
        return []
    return [re.sub(r"(?i)^and\s+", "", item.strip()) for item in re.split(r"[,;]\s*", value) if item.strip()]


def normalize_time_text(value: str | None) -> str | None:
    if not value:
        return None
    compact = re.sub(r"\s+", " ", value.strip()).upper()
    for fmt in ("%I:%M %p", "%I %p", "%H:%M"):
        try:
            return datetime.strptime(compact, fmt).strftime("%H:%M")
        except ValueError:
            pass
    return None


def parse_coordinate_component(value: str | float | int, axis: str) -> float:
    if axis not in {"latitude", "longitude"}:
        raise ValueError("axis must be 'latitude' or 'longitude'")
    if isinstance(value, (int, float)):
        result = float(value)
    else:
        text = value.strip().upper().replace("°", " ").replace("'", " ")
        hemisphere_match = re.search(r"([NSEW])$", text)
        hemisphere = hemisphere_match.group(1) if hemisphere_match else None
        if hemisphere:
            text = text[:hemisphere_match.start()].strip()
        parts = [part for part in re.split(r"[\s,]+", text) if part]
        if len(parts) == 1:
            result = float(parts[0])
        elif len(parts) == 2:
            degrees, minutes = map(float, parts)
            if not 0 <= abs(minutes) < 60:
                raise ValueError(f"invalid decimal minutes: {value!r}")
            result = abs(degrees) + abs(minutes) / 60
            if degrees < 0:
                result *= -1
        else:
            raise ValueError(f"unsupported coordinate format: {value!r}")
        if hemisphere:
            if axis == "latitude" and hemisphere not in {"N", "S"}:
                raise ValueError(f"invalid latitude hemisphere: {hemisphere}")
            if axis == "longitude" and hemisphere not in {"E", "W"}:
                raise ValueError(f"invalid longitude hemisphere: {hemisphere}")
            result = abs(result) * (-1 if hemisphere in {"S", "W"} else 1)
        elif axis == "longitude" and result > 0:
            raise ValueError("positive longitude is ambiguous without an E/W hemisphere")

    limit = 90 if axis == "latitude" else 180
    if not -limit <= result <= limit:
        raise ValueError(f"{axis} out of range: {result}")
    return result


assert parse_coordinate_component(48.12345, "latitude") == 48.12345
assert parse_coordinate_component(-123.45678, "longitude") == -123.45678
assert round(parse_coordinate_component("48 16.54 N", "latitude"), 6) == 48.275667
assert round(parse_coordinate_component("123 30.19 W", "longitude"), 6) == -123.503167
try:
    parse_coordinate_component("123 30.19", "longitude")
except ValueError:
    pass
else:
    raise AssertionError("positive longitude must not be silently interpreted as west")


## 5. Normalize a 5–20 record proof of concept

`source_record_id` is the Atlist marker UUID. `encounter_id` remains null because this reconnaissance has not established a durable CWR-wide identifier across website generations. The encounter number is parsed from the marker title. The map coordinate is retained separately from start/end coordinates.


In [ ]:
def normalize_marker(marker: dict[str, Any], expected_year: int = 2026) -> dict[str, Any]:
    name = marker.get("name") or ""
    name_match = ENCOUNTER_NAME_PATTERN.match(name)
    encounter_number = name_match.group("number").strip() if name_match else None
    date_source_text = name_match.group("date").strip() if name_match else None
    encounter_date = None
    if date_source_text:
        try:
            encounter_date = datetime.strptime(date_source_text, "%b %d, %Y").date().isoformat()
        except ValueError:
            pass

    note_fields = parse_labeled_notes(marker.get("notes"))
    tags = list(marker.get("tags") or [])
    ecotype = next((tag for tag in tags if tag in ECOTYPE_TAGS), None)
    pods = split_values(note_fields.get("Pods"))
    if not pods:
        pods = [tag.removesuffix(" Pod") for tag in tags if tag.endswith(" Pod")]

    map_lat = parse_coordinate_component(marker["lat"], "latitude") if marker.get("lat") is not None else None
    map_lon = parse_coordinate_component(marker["long"], "longitude") if marker.get("long") is not None else None
    qc_flags: list[str] = []
    if encounter_date and int(encounter_date[:4]) != expected_year:
        qc_flags.append("map_year_date_mismatch")
    if not encounter_number:
        qc_flags.append("missing_encounter_number")
    if map_lat is None or map_lon is None:
        qc_flags.append("missing_map_coordinate")

    return {
        "source": "center_for_whale_research",
        "source_system": "atlist",
        "source_record_id": marker.get("id"),
        "source_map_id": ATLIST_MAP_ID,
        "encounter_id": None,
        "encounter_number": encounter_number,
        "date": encounter_date,
        "date_source_text": date_source_text,
        "start_time": normalize_time_text(note_fields.get("ObservBegin")),
        "start_time_source_text": note_fields.get("ObservBegin") or None,
        "end_time": normalize_time_text(note_fields.get("ObservEnd")),
        "end_time_source_text": note_fields.get("ObservEnd") or None,
        "time_zone": None,
        "ecotype": ecotype,
        "pods": pods,
        "individuals": split_values(note_fields.get("IDsEncountered")),
        "location_description": note_fields.get("LocationDescr") or None,
        "map_lat": map_lat,
        "map_lon": map_lon,
        "coordinate_crs": "EPSG:4326",
        "coordinate_role": "map_marker_unspecified" if map_lat is not None and map_lon is not None else None,
        "start_lat": None,
        "start_lon": None,
        "end_lat": None,
        "end_lon": None,
        "summary": note_fields.get("EncSummary") or None,
        "source_url": TARGET_PAGE,
        "source_map_url": ATLIST_IFRAME_URL,
        "source_updated_at": marker.get("updatedAt"),
        "qc_flags": qc_flags,
    }


normalized_records = [normalize_marker(marker) for marker in markers]
normalized_records.sort(key=lambda row: (row["date"] or "9999-99-99", row["encounter_number"] or ""))
poc_records = normalized_records[:POC_LIMIT]
assert 5 <= len(poc_records) <= 20

preview_columns = [
    "source_record_id", "encounter_number", "date", "start_time", "end_time",
    "ecotype", "pods", "individuals", "location_description",
    "map_lat", "map_lon", "coordinate_role", "qc_flags",
]
display(pd.DataFrame(poc_records)[preview_columns])
poc_json_preview = []
for record in poc_records:
    item = dict(record)
    if item.get("summary") and len(item["summary"]) > 300:
        item["summary"] = item["summary"][:300].rstrip() + "… <truncated for display>"
    poc_json_preview.append(item)
print(json.dumps(poc_json_preview, indent=2, ensure_ascii=False))


In [ ]:
def sanitized_structural_sample(records: list[dict[str, Any]]) -> dict[str, Any]:
    # Preserve structure and provenance while omitting narrative and individual-whale content.
    sample_records = []
    for record in records[:5]:
        item = dict(record)
        item["summary"] = "<omitted from sanitized sample>" if item.get("summary") else None
        item["individuals"] = ["<omitted from sanitized sample>"] if item.get("individuals") else []
        sample_records.append(item)
    return {
        "sample_kind": "sanitized_normalized_poc",
        "source_endpoint": ATLIST_ENDPOINTS["markers"],
        "map_updated_at": map_metadata.get("updatedAt") if isinstance(map_metadata, dict) else None,
        "records": sample_records,
    }


if WRITE_SANITIZED_SAMPLE:
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    sample_path = SAMPLE_DIR / "atlist_2026_normalized_sample.sanitized.json"
    sample_path.write_text(json.dumps(sanitized_structural_sample(poc_records), indent=2, ensure_ascii=False) + "\n")
    print("Wrote", sample_path.resolve())
else:
    print("Sanitized sample writing is disabled; set WRITE_SANITIZED_SAMPLE = True to opt in.")


## 6. Quality-control reconnaissance

These checks surface rather than repair source anomalies: missing/variable note fields, unexpected years, duplicate numbers, coordinate bounds, and source record ID uniqueness.


In [ ]:
records_df = pd.DataFrame(normalized_records)
field_availability = pd.DataFrame({
    "field": records_df.columns,
    "non_null_count": [records_df[column].notna().sum() for column in records_df.columns],
    "total_records": len(records_df),
})
field_availability["non_null_fraction"] = field_availability["non_null_count"] / field_availability["total_records"].clip(lower=1)

qc_summary = {
    "record_count": len(records_df),
    "unique_source_record_ids": records_df["source_record_id"].nunique(dropna=True),
    "duplicate_source_record_ids": int(records_df["source_record_id"].duplicated().sum()),
    "missing_dates": int(records_df["date"].isna().sum()),
    "map_year_mismatches": int(records_df["qc_flags"].apply(lambda flags: "map_year_date_mismatch" in flags).sum()),
    "invalid_latitudes": int((~records_df["map_lat"].between(-90, 90)).fillna(False).sum()),
    "invalid_longitudes": int((~records_df["map_lon"].between(-180, 180)).fillna(False).sum()),
}
display(qc_summary)
display(field_availability.sort_values(["non_null_fraction", "field"]))
display(records_df.loc[records_df["qc_flags"].map(bool), ["source_record_id", "encounter_number", "date", "qc_flags"]])


## 7. Detail pages and historical website generations

The current Atlist markers expose rich notes directly, but this does not establish whether CWR publishes separate 2026 detail pages. The CWR navigation currently exposes distinct entry points for 2025, 2024, 2023, and an archive site. Inspect each page independently rather than extrapolating a URL pattern.


In [ ]:
YEAR_ENTRY_POINTS = {
    2026: TARGET_PAGE,
    2025: "https://www.whaleresearch.com/encounters",
    2024: "https://www.whaleresearch.com/encounters2024",
    2023: "https://www.whaleresearch.com/encounters2023",
    "archive": "https://whaleresearch.wixsite.com/archives",
}

historical_inventory = []
for year, url in YEAR_ENTRY_POINTS.items():
    try:
        _, inventory = inspect_page_html(url)
        historical_inventory.append({
            "year": year,
            "url": url,
            "status": inventory.get("status"),
            "iframes": inventory.get("iframes", []),
            "json_script_ids": inventory.get("json_script_ids", []),
        })
    except requests.RequestException as exc:
        historical_inventory.append({"year": year, "url": url, "error": repr(exc)})

display(pd.DataFrame(historical_inventory))


In [ ]:
schema_comparison = pd.DataFrame(
    [
        ("source_record_id", True, "not yet established", "not yet established"),
        ("encounter_number", True, "not yet established", "not yet established"),
        ("date", True, "not yet established", "not yet established"),
        ("start_time", any(row["start_time"] for row in normalized_records), "not yet established", "not yet established"),
        ("end_time", any(row["end_time"] for row in normalized_records), "not yet established", "not yet established"),
        ("ecotype", any(row["ecotype"] for row in normalized_records), "not yet established", "not yet established"),
        ("pods", any(row["pods"] for row in normalized_records), "not yet established", "not yet established"),
        ("individuals", any(row["individuals"] for row in normalized_records), "not yet established", "not yet established"),
        ("location_description", any(row["location_description"] for row in normalized_records), "not yet established", "not yet established"),
        ("map_lat/map_lon", all(row["map_lat"] is not None and row["map_lon"] is not None for row in normalized_records), "not yet established", "not yet established"),
        ("start_lat/start_lon", False, "not yet established", "not yet established"),
        ("end_lat/end_lon", False, "not yet established", "not yet established"),
        ("summary", any(row["summary"] for row in normalized_records), "not yet established", "not yet established"),
    ],
    columns=["field", "2026 Atlist map", "2026 detail", "historical detail"],
)
display(schema_comparison)


## Provisional recommendation

For the 2026 map, prefer a bounded direct GET of the public Atlist `/markers` endpoint over DOM scraping or recurring Playwright automation. It is simpler, returns stable source UUIDs and signed decimal coordinates, and does not currently require authentication, cookies, a request body, or observed pagination. Use Playwright as a discovery/regression tool to rediscover the endpoint when the embed changes—not as the normal extractor.

A later production design should:

1. snapshot the raw response immutably with retrieval time, URL, map ID, source update timestamps, checksum, and HTTP metadata;
2. retain raw marker title, HTML notes, tags, UUID, category ID, and the single map coordinate before normalization;
3. parse labeled note fields while preserving missing/null status and source text;
4. keep the map point distinct from start/end coordinates and preserve unknown time zone;
5. add explicit QC for source-year mismatches, missing labels, duplicate IDs/numbers, and changing field coverage;
6. investigate 2026 detail links and historical website generations separately before defining a cross-year encounter identity; and
7. confirm permission/licensing for recurring ingestion and redistribution, especially for narratives, photos, and individual-whale details.

### Remaining reconnaissance

- Click representative Atlist markers and confirm whether any button or note links to a CWR detail page.
- Inspect several recent and archival records and replace `not yet established` values in the schema comparison with evidence.
- Determine whether historical maps are also Atlist maps with independent map IDs or older Wix/HTML record generations.
- Verify that map coordinates mean observation location rather than a manually chosen display point.
- Record the site's permission/licensing outcome before production use.
